# Exercise 1 Solution: Creating Embeddings

In [ ]:
import json
import sys
from pathlib import Path
import asyncio
import numpy as np
from sentence_transformers import SentenceTransformer

REPO_ROOT = Path('..')
sys.path.insert(0, str(REPO_ROOT / 'src'))
from models import PostDocument
from source import load_sample_posts, convert_to_pydantic, create_embeddings_for_posts

model = SentenceTransformer('all-MiniLM-L6-v2')

## Task 1 — Encode a single sentence

In [ ]:
sentence = "Cats are amazing animals."
embedding = model.encode(sentence)  # shape (384,)
print(f"Shape:    {embedding.shape}")
print(f"Dtype:    {embedding.dtype}")
print(f"First 5 values: {embedding[:5]}")
assert embedding.shape == (384,)
print("✓ Task 1 complete")

## Task 2 — Sentence embeddings → document embedding

In [ ]:
with open(REPO_ROOT / 'sample_posts.json') as f:
    raw_posts = json.load(f)
post = PostDocument(**raw_posts[0])
sentences = post.preprocess_sentences()

sentence_embeddings = model.encode(sentences)            # (n_sentences, 384)
doc_embedding = np.mean(sentence_embeddings, axis=0)     # (384,)

print(f"Sentence embeddings shape: {sentence_embeddings.shape}")
print(f"Document embedding shape:  {doc_embedding.shape}")
assert doc_embedding.shape == (384,)
print("✓ Task 2 complete")

## Task 3 — Embed the full corpus

In [ ]:
raw = load_sample_posts(str(REPO_ROOT / 'sample_posts.json'))
posts = convert_to_pydantic(raw)

await create_embeddings_for_posts(posts)

assert all(len(p.doc_embedding) == 384 for p in posts)
print(f"✓ Task 3 complete — {len(posts)} posts embedded, dim={len(posts[0].doc_embedding)}")